In [27]:
# Standard libraries
import os
from pathlib import Path
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [28]:
df_80 = pd.read_csv('Status_1_region_2.csv')
df_81 = pd.read_csv('Status_1_region_3.csv')
df_85 = pd.read_csv('Status_1_region.csv')

In [29]:
df_85

,SamplingOperations_code,yhat_te
0,S05169000_20130911,High
1,S05169000_20200819,High
2,S05172000_20210813,Good
3,S05172050_20210824,High
4,S05172350_20120827,High
...,...,...
167,S06175540_20150728,High
168,S06175600_20170904,High
169,S06175600_20180926,Good
170,S06175645_20220809,Good


In [30]:
n = len(df_80)

In [31]:
df_80_81 = pd.merge(df_80,df_81,on='SamplingOperations_code')

In [32]:
df_80_81

,SamplingOperations_code,yhat_te_x,yhat_te_y
0,S05169000_20130911,High,High
1,S05169000_20200819,Good,Good
2,S05172000_20210813,Good,Good
3,S05172050_20210824,Moderate,Good
4,S05172350_20120827,High,High
...,...,...,...
167,S06175540_20150728,High,High
168,S06175600_20170904,High,High
169,S06175600_20180926,High,Good
170,S06175645_20220809,High,High


In [33]:
df_80

,SamplingOperations_code,yhat_te
0,S05169000_20130911,High
1,S05169000_20200819,Good
2,S05172000_20210813,Good
3,S05172050_20210824,Moderate
4,S05172350_20120827,High
...,...,...
167,S06175540_20150728,High
168,S06175600_20170904,High
169,S06175600_20180926,High
170,S06175645_20220809,High


In [35]:
import pandas as pd

# Cargar las predicciones
a = df_80
b = df_81
c = df_85
 
# Unir por id
df = a.merge(b, on="SamplingOperations_code", suffixes=("_81", "_80"))
df = df.merge(c, on="SamplingOperations_code", suffixes=("", "_85"))
df.rename(columns={"yhat_te": "yhat_te_85"}, inplace=True)
df

,SamplingOperations_code,yhat_te_81,yhat_te_80,yhat_te_85
0,S05169000_20130911,High,High,High
1,S05169000_20200819,Good,Good,High
2,S05172000_20210813,Good,Good,Good
3,S05172050_20210824,Moderate,Good,High
4,S05172350_20120827,High,High,High
...,...,...,...,...
167,S06175540_20150728,High,High,High
168,S06175600_20170904,High,High,High
169,S06175600_20180926,High,Good,Good
170,S06175645_20220809,High,High,Good


In [37]:
# Votación mayoritaria
df["consenso"] = df[["yhat_te_81", "yhat_te_80", "yhat_te_85"]].mode(axis=1)[0]

In [38]:
df

,SamplingOperations_code,yhat_te_81,yhat_te_80,yhat_te_85,consenso
0,S05169000_20130911,High,High,High,High
1,S05169000_20200819,Good,Good,High,Good
2,S05172000_20210813,Good,Good,Good,Good
3,S05172050_20210824,Moderate,Good,High,Good
4,S05172350_20120827,High,High,High,High
...,...,...,...,...,...
167,S06175540_20150728,High,High,High,High
168,S06175600_20170904,High,High,High,High
169,S06175600_20180926,High,Good,Good,Good
170,S06175645_20220809,High,High,Good,High


In [39]:
# Coincidencia total (los 3 iguales)
df["coinciden_todos"] = (
    (df["yhat_te_81"] == df["yhat_te_80"]) & 
    (df["yhat_te_81"] == df["yhat_te_85"])
)


In [40]:




# Proporción de coincidencia
p_coincidencia = df["coinciden_todos"].mean()

print(f"Coincidencia total entre los 3 modelos: {p_coincidencia:.2%}")

# Guardar la predicción final (de consenso)
pred_coincidecnias = df[["SamplingOperations_code", "consenso"]]

Coincidencia total entre los 3 modelos: 87.21%


In [41]:
pred_coincidecnias

,SamplingOperations_code,consenso
0,S05169000_20130911,High
1,S05169000_20200819,Good
2,S05172000_20210813,Good
3,S05172050_20210824,Good
4,S05172350_20120827,High
...,...,...
167,S06175540_20150728,High
168,S06175600_20170904,High
169,S06175600_20180926,Good
170,S06175645_20220809,High


In [43]:
# Nombres de columnas (ajústalos si cambian)
col_id = "SamplingOperations_code"
col_a  = "yhat_te_81"  # Modelo A (81%)
col_b  = "yhat_te_80"  # Modelo B (80%)
col_c  = "yhat_te_85"  # Modelo C (85%)

# =========================
# 1) Coincidencias par a par
# =========================
df["coincide_ab"] = df[col_a] == df[col_b]
df["coincide_ac"] = df[col_a] == df[col_c]
df["coincide_bc"] = df[col_b] == df[col_c]

# DFs filtrados SOLO con coincidencias (si prefieres ver solo los que sí coinciden)
df_ab = df.loc[df["coincide_ab"], [col_id, col_a, col_b]].copy()
df_ac = df.loc[df["coincide_ac"], [col_id, col_a, col_c]].copy()
df_bc = df.loc[df["coincide_bc"], [col_id, col_b, col_c]].copy()

# (opcional) también puedes quedarte con el flag en versión "completa":
# df_ab_full = df[[col_id, col_a, col_b, "coincide_ab"]].copy()
# df_ac_full = df[[col_id, col_a, col_c, "coincide_ac"]].copy()
# df_bc_full = df[[col_id, col_b, col_c, "coincide_bc"]].copy()

# =========================
# 2) Triple coincidencia
# =========================
df["coinciden_todos"] = (df[col_a] == df[col_b]) & (df[col_a] == df[col_c])
df_triple = df.loc[df["coinciden_todos"], [col_id, col_a, col_b, col_c]].copy()

# (métrica) proporción de triple coincidencia
p_triple = df["coinciden_todos"].mean()
print(f"Triple coincidencia (A=B=C): {p_triple:.2%}")

# =========================
# 3) Votación mayoritaria + % de coincidencia con A, B y C
# =========================
# Predicción de consenso por mayoría
df["consenso"] = df[[col_a, col_b, col_c]].mode(axis=1)[0]

# % de acuerdo del consenso con cada modelo
p_consenso_con_a = (df["consenso"] == df[col_a]).mean()
p_consenso_con_b = (df["consenso"] == df[col_b]).mean()
p_consenso_con_c = (df["consenso"] == df[col_c]).mean()

print(f"Consenso vs A (81%): {p_consenso_con_a:.2%}")
print(f"Consenso vs B (80%): {p_consenso_con_b:.2%}")
print(f"Consenso vs C (85%): {p_consenso_con_c:.2%}")

# DF final de consenso (id + predicción)
df_consenso = df[[col_id, "consenso"]].copy()

# =========================
# (Opcional) Guardados
# =========================
# df_ab.to_csv("results/coincidencias_ab.csv", index=False)
# df_ac.to_csv("results/coincidencias_ac.csv", index=False)
# df_bc.to_csv("results/coincidencias_bc.csv", index=False)
# df_triple.to_csv("results/triple_coincidencia.csv", index=False)
# df_consenso.to_csv("results/prediccion_consenso.csv", index=False)


Triple coincidencia (A=B=C): 87.21%
Consenso vs A (81%): 95.93%
Consenso vs B (80%): 97.09%
Consenso vs C (85%): 93.60%


In [44]:
print(len(df_ab)/n)
print(len(df_bc)/n)
print(len(df_ac)/n)

0.9302325581395349
0.9069767441860465
0.9011627906976745
